<div align="center">

<a href="https://colab.research.google.com/github/utkukose/philosophy-of-ai-NB-lecture/blob/main/weeks/week-08/NB08_frame_problem.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open in Colab"></a>

# Week 08: The Frame Problem and Common Sense

**Philosophy of Artificial Intelligence (11118BLG003)**<br>Prof. Dr. Utku Kose<br>Full Professor, Department of Computer Engineering, Süleyman Demirel University, Isparta, Türkiye<br>Additional affiliations: University of North Dakota (USA), Universidad Panamericana (Mexico City, Mexico), Vel Tech University (Chennai, India)<br>[ORCID 0000-0002-9652-6415](https://orcid.org/0000-0002-9652-6415) | [utkukose.com](https://www.utkukose.com) | [github.com/utkukose](https://github.com/utkukose)

*This course is updated in line with current developments in the field. Last update: September 2026.*

</div>

## How to use this notebook

This notebook is a self-contained version of the week: the lecture text comes first, followed by the hands-on lab. Run the cells in order with Shift+Enter. Exercise cells start with `None` placeholders and print feedback through `check(...)`. Reference solutions sit in collapsed cells: open them only after a genuine attempt. The interactive lab and the self-assessment are available at <https://utkukose.github.io/philosophy-of-ai-NB-lecture/weeks/week-08/lab.html>.

## Overview

An intelligent agent must know what changes when it acts and what stays the same, and it must recognise which of its countless beliefs matter. This week traces the frame problem from its logical origin to its philosophical generalisation as a problem of relevance, and connects it with the challenge of common sense [1, 2, 3].

## Learning outcomes

By the end of the week, students are expected to distinguish the representational frame problem from the general problem of relevance, to explain STRIPS and successor state axioms as solutions to the former, to reconstruct Dennett's robot story, and to explain why Winograd schemas test common sense.

## A logical problem

McCarthy and Hayes introduced the situation calculus to represent actions and their effects in logic, and noticed a difficulty [1]. To reason that painting a block does not move it, the system needs an explicit axiom saying so, and the same holds for every pair of an action and a property it does not affect. The number of these frame axioms grows with the product of actions and properties, called fluents. Figure 8.1 shows the growth. They named this the frame problem.

Two classic answers address the representational side. The STRIPS planner of Fikes and Nilsson represented each action by lists of facts it adds and deletes and assumed that everything else persists [4]. Reiter later showed how to compile all frame axioms into one successor state axiom per fluent, which states exactly when the fluent is true after an action [5]. Shanahan presented a detailed treatment of such solutions based on a common sense law of inertia [6]. In the logical sense, the frame problem is largely solved.

![Figure 8.1](https://raw.githubusercontent.com/utkukose/philosophy-of-ai-NB-lecture/main/weeks/week-08/figures/w08_fig1.png)

*Figure 8.1. Explicit frame axioms grow with the product of actions and fluents, while successor state axioms grow with the number of fluents [1, 5].*

## A problem of relevance

Dennett told a story that turned the frame problem into a general problem of cognition [2]. A robot, R1, must fetch its spare battery from a room in which a bomb is timed to go off. The battery is on a wagon, and so is the bomb. R1 pulls the wagon out, knowing that the battery is on it, but does not notice that the bomb comes along too. Its designers build R1D1, which deduces the implications of its actions, and it is still deducing irrelevant consequences, such as that pulling the wagon would not change the colour of the walls, when the bomb explodes. The next model, R2D1, is taught to ignore irrelevant implications, but it is busy classifying thousands of them as irrelevant when time runs out.

The moral is that the difficulty lies not in representing change but in determining relevance quickly, without considering everything. People do this effortlessly, and no one knows how. Dreyfus, whose critique is examined in Week 9, saw in the problem evidence that intelligence rests on background coping rather than on rules.

## Common sense

Davis and Marcus surveyed the challenge of commonsense reasoning and argued that it remains a central obstacle for AI [3]. Commonsense knowledge is vast, mostly unstated and used without effort: that objects fall when dropped, that containers hold things smaller than themselves, that people act on their beliefs. Levesque, Davis and Morgenstern proposed the Winograd schema challenge as a test [7]. Each item is a pair of sentences that differ in one word, which flips the referent of a pronoun. In the pair about a trophy that does not fit into a suitcase because it is too large or too small, the pronoun refers to the trophy in one case and to the suitcase in the other. Surface statistics that ignore the special word can do no better than chance.

## Relevance in present systems

Large language models answer many Winograd-style questions correctly, which shows that statistics over vast text capture much commonsense knowledge. Whether they solve the relevance problem or sidestep it is debated: A model that has absorbed enormous numbers of situations may retrieve the relevant pattern without ever facing Dennett's explosion, but it may also fail on situations that its training did not cover. The lab reconstructs Dennett's robots with an adjustable time budget, and the notebook counts frame axioms and tests surface heuristics on Winograd-style pairs.

> **Pause and reflect.** When you decide what to check before leaving home, how do you avoid considering irrelevant facts? Could that procedure be written down?

# Hands-on lab

The notebook grounds a blocks world in STRIPS operators, counts the frame axioms that an explicit situation calculus encoding would need, and tests two surface heuristics on Winograd-style sentence pairs [4, 5, 7].

In [ ]:
# Setup: common libraries and the self-check helper used in the exercises.
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})
SEED = 2026

def check(name, value, expected, tol=1e-6):
    """Compare an exercise answer with the reference value and print feedback."""
    if value is None:
        print(f"[{name}] Not attempted yet. Replace None with your answer and run the cell again.")
        return
    try:
        ok = bool(np.allclose(np.asarray(value, dtype=float), np.asarray(expected, dtype=float), atol=tol))
    except (TypeError, ValueError):
        ok = value == expected
    print(f"[{name}] " + ("Correct." if ok else "Not yet. Revisit the explanation above and try again."))

## 1. STRIPS operators for a blocks world

In [ ]:
import itertools

def blocks_world(blocks):
    fluents = [f"on({a},{b})" for a in blocks for b in blocks if a != b] + [f"ontable({a})" for a in blocks] + \
              [f"clear({a})" for a in blocks] + [f"holding({a})" for a in blocks] + ["handempty"]
    actions = {}
    for a in blocks:
        actions[f"pickup({a})"] = ({f"holding({a})"}, {f"ontable({a})", f"clear({a})", "handempty"})
        actions[f"putdown({a})"] = ({f"ontable({a})", f"clear({a})", "handempty"}, {f"holding({a})"})
        for b in blocks:
            if a != b:
                actions[f"stack({a},{b})"] = ({f"on({a},{b})", f"clear({a})", "handempty"}, {f"holding({a})", f"clear({b})"})
                actions[f"unstack({a},{b})"] = ({f"holding({a})", f"clear({b})"}, {f"on({a},{b})", f"clear({a})", "handempty"})
    return fluents, actions

fluents, actions = blocks_world(["A", "B", "C"])
print(len(fluents), "fluents,", len(actions), "grounded actions")

## 2. Counting frame axioms

An explicit encoding needs one frame axiom for every pair of an action and a fluent that the action leaves unchanged. STRIPS avoids them by assuming persistence, and successor state axioms need one axiom per fluent.

### Your turn, exercise 1

Write `frame_axioms(fluents, actions)` that counts the pairs (action, fluent) in which the fluent is neither added nor deleted by the action. Then compare the count for three and for five blocks with the number of fluents.

In [ ]:
def frame_axioms(fluents, actions):
    # Your code here
    return None

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
def _frame_reference(fluents, actions):
    return sum(len([f for f in fluents if f not in add | delete]) for add, delete in actions.values())

In [ ]:
check("Exercise 1", frame_axioms(fluents, actions), _frame_reference(fluents, actions))
for k in [3, 4, 5, 6]:
    fl, ac = blocks_world([chr(65 + i) for i in range(k)])
    print(f"{k} blocks: {len(fl)} fluents, {len(ac)} actions, {_frame_reference(fl, ac)} explicit frame axioms")

## 3. Winograd-style pairs

Each pair differs in one word, which flips the referent of the pronoun. The first four pairs follow the pattern of classic examples; the others were written for this notebook.

In [ ]:
PAIRS = [
    ("The trophy does not fit into the suitcase because it is too large.", ("trophy", "suitcase"), 0),
    ("The trophy does not fit into the suitcase because it is too small.", ("trophy", "suitcase"), 1),
    ("The council refused the demonstrators a permit because they feared violence.", ("council", "demonstrators"), 0),
    ("The council refused the demonstrators a permit because they advocated violence.", ("council", "demonstrators"), 1),
    ("The ball broke the table because it was made of steel.", ("ball", "table"), 0),
    ("The ball broke the table because it was made of glass.", ("ball", "table"), 1),
    ("The fish ate the worm because it was hungry.", ("fish", "worm"), 0),
    ("The fish ate the worm because it was tasty.", ("fish", "worm"), 1),
    ("The man could not lift his son because he was so weak.", ("man", "son"), 0),
    ("The man could not lift his son because he was so heavy.", ("man", "son"), 1),
]
first_candidate = np.mean([gold == 0 for _, _, gold in PAIRS])
print("always choose the first candidate:", first_candidate)

### Your turn, exercise 2

Compute the accuracy of the heuristic that always chooses the candidate mentioned closest to the pronoun (the second candidate). Explain in two sentences why every heuristic that ignores the special word must score 0.5 on twin pairs.

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
def _recent_reference():
    return float(np.mean([gold == 1 for _, _, gold in PAIRS]))

In [ ]:
recent_accuracy = None  # your computation
check("Exercise 2", recent_accuracy, _recent_reference())

## Visual exploration: The cost of relevance

A robot that checks every fact after each action pays a cost that grows with its world model. A learned relevance score is cheap but can miss affected facts. The panels compare the two strategies as the world model grows.

In [ ]:
rng = np.random.default_rng(SEED)
sizes, rows = [10, 30, 100, 300, 1000, 3000, 10000], []
for n_facts in sizes:
    relevant = rng.random(n_facts) < 0.02                         # facts actually affected by the action
    score = relevant * 1.0 + rng.normal(0, 0.35, n_facts)          # a learned relevance score
    checked = score > 0.5
    rows.append((n_facts, checked.sum(), np.sum(relevant & ~checked) / max(relevant.sum(), 1)))
rows = np.array(rows, dtype=float)
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].loglog(rows[:, 0], rows[:, 0], "o-", label="check every fact"); ax[0].loglog(rows[:, 0], np.maximum(rows[:, 1], 1), "s-", label="check facts scored as relevant")
ax[0].set_xlabel("facts in the world model"); ax[0].set_ylabel("facts checked per action"); ax[0].legend()
ax[1].semilogx(rows[:, 0], rows[:, 2], "o-"); ax[1].set_xlabel("facts in the world model"); ax[1].set_ylabel("share of relevant facts missed")
fig.suptitle("Relevance filtering is cheap but can miss what matters"); plt.tight_layout(); plt.show()

## Self-assessment and reflection

Take the self-assessment in the interactive lab: <https://utkukose.github.io/philosophy-of-ai-NB-lecture/weeks/week-08/lab.html> (tab: Check yourself). Then answer the reflection prompts:

1. In the lab, how did you know which facts to tick? Could that knowledge be written as rules?
2. Do large language models solve the relevance problem, sidestep it or merely hide it? Give an example that supports your answer.
3. Describe a failure of common sense that you have seen in an AI system and explain it with the ideas of this week.

## Weekly task and submission

Write about 500 words distinguishing the logical frame problem from the problem of relevance, using the lab and the notebook as examples, and assess whether present language models face the second problem. Attach the notebook with both exercises completed.

The weekly task supports self-learning and builds a personal portfolio. When the course is followed with the instructor during an active semester, the task can be sent together with the exported learning log to utkukose@sdu.edu.tr or utkukose@gmail.com for evaluation.

## Research and report assignment (optional)

**Commonsense benchmarks after Winograd.** Review what happened to Winograd-style benchmarks once language models approached human scores, and what this shows about common sense and relevance [2, 3, 7].

This research assignment is optional and supports self-learning. When the related weeks are followed within the course during an active semester, the report can be sent to utkukose@sdu.edu.tr or utkukose@gmail.com for evaluation. Unless the assignment states otherwise, a report has 1500 to 2500 words, follows the structure of an academic paper, cites at least six scholarly or official sources in square brackets and ends with a reference list.

## References

[1] McCarthy, J., & Hayes, P. J. (1969). Some philosophical problems from the standpoint of artificial intelligence. In B. Meltzer & D. Michie (Eds.), *Machine Intelligence 4* (pp. 463-502). Edinburgh University Press.

[2] Dennett, D. C. (1984). Cognitive wheels: The frame problem of AI. In C. Hookway (Eds.), *Minds, Machines and Evolution* (pp. 129-151). Cambridge University Press.

[3] Davis, E., & Marcus, G. (2015). Commonsense reasoning and commonsense knowledge in artificial intelligence. *Communications of the ACM*, 58(9), 92-103. <https://doi.org/10.1145/2701413>

[4] Fikes, R. E., & Nilsson, N. J. (1971). STRIPS: A new approach to the application of theorem proving to problem solving. *Artificial Intelligence*, 2(3-4), 189-208. <https://doi.org/10.1016/0004-3702(71)90010-5>

[5] Reiter, R. (2001). *Knowledge in Action: Logical Foundations for Specifying and Implementing Dynamical Systems*. MIT Press.

[6] Shanahan, M. (1997). *Solving the Frame Problem: A Mathematical Investigation of the Common Sense Law of Inertia*. MIT Press.

[7] Levesque, H. J., Davis, E., & Morgenstern, L. (2012). The Winograd schema challenge. In *Proceedings of the 13th International Conference on Principles of Knowledge Representation and Reasoning (KR 2012)* (pp. 552-561). AAAI Press.